# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read bronze Table

In [0]:
df =spark.table("bikescatalog.bronze.erp_px_cat_g1v2")

In [0]:
df.display()

# Silver Transformations

# Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(field.name))

# Normalize Maintance Flag to Boolean

In [0]:
df = df.withColumn(
        "Maintenance",
        F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
         .when(F.upper(col("MAINTENANCE")) == "NO", F.lit(False))
         .otherwise(None)
        
)

# Renaming columns

In [0]:
RENAME_MAP = {
    "id" : "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Sanity check of dataframe

In [0]:
df.limit(10).display()

# Writing silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("bikescatalog.silver.erp_product_category")

# Sanity check of silver table

In [0]:
%sql
select * from bikescatalog.silver.erp_product_category limit 10